In [1]:
from pathlib import Path
import json
import os
import re
from datetime import datetime

IMAGE_DIR = Path("data")
QWEN_MODEL_DIR = Path("models/qwen3_vl_2b_instruct")
QWEN_MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TRANSLATION_MODEL = os.getenv("TRANSLATION_MODEL", "Helsinki-NLP/opus-mt-en-ru")
OUTPUT_JSON = Path("dataset.json")
OUTPUT_JSONL = Path("dataset.jsonl")
MAX_IMAGES = int(os.getenv("MAX_IMAGES", "0")) or None

image_files = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)
if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]

if not QWEN_MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Qwen model directory not found: {QWEN_MODEL_DIR}")

print("Images:", len(image_files))
print("Qwen model:", QWEN_MODEL_DIR)
print("Translation model:", TRANSLATION_MODEL)


Images: 174
Qwen model: models\qwen3_vl_2b_instruct
Translation model: Helsinki-NLP/opus-mt-en-ru


In [2]:
import torch
from PIL import Image
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    MarianMTModel,
    MarianTokenizer,
)

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)

processor = AutoProcessor.from_pretrained(QWEN_MODEL_DIR, local_files_only=True)
qwen_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_DIR,
    local_files_only=True,
    torch_dtype="auto",
)
qwen_model.eval()

translator_tokenizer = MarianTokenizer.from_pretrained(TRANSLATION_MODEL)
translator_model = MarianMTModel.from_pretrained(TRANSLATION_MODEL)
translator_model.eval()

print("Models loaded")


Torch device: cpu


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Models loaded


In [3]:
def clean_text(text):
    return " ".join((text or "").strip().split()).strip('"')


def clean_qwen_caption(text):
    text = clean_text(text)
    text = re.sub(r"^(The image shows|This image shows|In the image,?)\s*", "", text, flags=re.I).strip()
    return text[:1].upper() + text[1:] if text else text


def describe_image_with_qwen(image_path):
    prompt = (
        "Describe the image in one concise English sentence. "
        "Mention the main objects, visible action, and scene. "
        "Do not invent details that are not visible."
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": prompt},
        ],
    }]
    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        inputs = processor(
            text=[chat_text],
            images=[image],
            return_tensors="pt",
        )
    with torch.no_grad():
        generated = qwen_model.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
        )
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    caption = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return clean_qwen_caption(caption)


def translate_en_ru(texts, batch_size=16):
    translations = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = translator_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
            )
            generated = translator_model.generate(
                **encoded,
                max_new_tokens=100,
                num_beams=4,
            )
            decoded = translator_tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
            )
            translations.extend(clean_text(text) for text in decoded)
    return translations

sample_caption = describe_image_with_qwen(image_files[0])
sample_text = translate_en_ru([sample_caption])[0]
print(image_files[0].name)
print("QWEN_EN:", sample_caption)
print("RU:", sample_text)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-01-16-2-3-2-5-15_jpg.rf.3a82c0e7cb6b006df4c9fc87d9b55368.jpg
QWEN_EN: A woman in a black shirt and light blue jeans is bending over to pick up an object from a black table in an office space, with several cardboard boxes stacked on a table and cubicles in the background.
RU: Женщина в черной рубашке и светло-голубых джинсах изгибается, чтобы забрать предмет из черного стола в офисном помещении, с несколькими картонными коробками, сложенными на столе, и кабинками на заднем плане.


In [4]:
def load_existing_items():
    if not OUTPUT_JSON.exists():
        return []
    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else []


def save_items(items):
    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

existing_items = load_existing_items()
by_image = {item.get("image"): item for item in existing_items}
items = []
pending_paths = []

for image_path in image_files:
    rel_path = image_path.as_posix()
    item = by_image.get(rel_path)
    if (
        item
        and item.get("caption_model") == QWEN_MODEL_NAME
        and item.get("qwen_caption_en")
        and item.get("text")
    ):
        items.append(item)
    else:
        pending_paths.append(image_path)

print("Existing Qwen records:", len(items))
print("Pending Qwen records:", len(pending_paths))

new_items = []
new_captions = []
for idx, image_path in enumerate(pending_paths, start=1):
    print(f"[{idx}/{len(pending_paths)}] qwen {image_path.name}")
    caption_en = describe_image_with_qwen(image_path)
    new_captions.append(caption_en)
    new_items.append({
        "image": image_path.as_posix(),
        "qwen_caption_en": caption_en,
        "caption_model": QWEN_MODEL_NAME,
        "caption_model_path": QWEN_MODEL_DIR.as_posix(),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })
    if idx % 5 == 0:
        save_items(items + new_items)
        print("saved partial", len(items) + len(new_items))

if new_items:
    print("Translating new captions to Russian")
    translations = translate_en_ru(new_captions)
    for item, text_ru in zip(new_items, translations):
        item["text"] = text_ru
        item["translation_model"] = TRANSLATION_MODEL
    by_image.update({item["image"]: item for item in new_items})
    items = [by_image[p.as_posix()] for p in image_files]
else:
    items = [by_image[p.as_posix()] for p in image_files]

save_items(items)
print(f"Saved {len(items)} items to {OUTPUT_JSON}")


Existing Qwen records: 174
Pending Qwen records: 0
Saved 174 items to dataset.json


In [5]:
with OUTPUT_JSON.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

print("Records:", len(dataset))
print("Expected images:", len(image_files))

for item in dataset[:5]:
    print("\nIMAGE:", item["image"])
    print("TEXT:", item["text"])
    print("QWEN_EN:", item.get("qwen_caption_en"))

if len(dataset) != len(image_files):
    raise RuntimeError(f"dataset.json has {len(dataset)} records; expected {len(image_files)}")


Records: 174
Expected images: 174

IMAGE: data/-01-16-2-3-2-5-15_jpg.rf.3a82c0e7cb6b006df4c9fc87d9b55368.jpg
TEXT: Женщина в черной рубашке и светло-голубых джинсах изгибается, чтобы забрать предмет из черного стола в офисном помещении, с несколькими картонными коробками, сложенными на столе, и кабинками на заднем плане.
QWEN_EN: A woman in a black shirt and light blue jeans is bending over to pick up an object from a black table in an office space, with several cardboard boxes stacked on a table and cubicles in the background.

IMAGE: data/-01-16-2-3-2-5-15_jpg.rf.705a29a51fa44f15dbe29adcaf09d2e0.jpg
TEXT: Человек с рыжими волосами изгибается, чтобы забрать объект из низкого стола в офисном помещении, с несколькими сложенными картонными коробками и шкафами, видимыми на заднем фоне.
QWEN_EN: A person with red hair is bending over to pick up an object from a low table in an office space, with several stacked cardboard boxes and cubicles visible in the background.

IMAGE: data/-01-16-2-3